# House Prices — Exploratory Data Analysis

This notebook explores the Kaggle House Prices dataset before any modelling.
The goal is to understand the data, find patterns, and make informed decisions about preprocessing.

Dataset: [House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Loading the Data

First, let's see what we're working with.

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

In [ ]:
train.head()

In [ ]:
train.info()

81 columns — 43 object (categorical), 35 int, 3 float. That's a lot of categorical columns to deal with.

Let's look at summary statistics for the numeric columns.

In [ ]:
train.describe()

## 2. Target Variable — SalePrice

Before anything else, let's understand what we're trying to predict.

In [ ]:
sns.histplot(train['SalePrice'], kde=True)
plt.title('SalePrice distribution')
plt.xlabel('SalePrice ($)')
plt.show()

print(train['SalePrice'].describe())
print(f'\nSkewness: {train["SalePrice"].skew():.3f}')

The distribution is right-skewed — most houses sell between \$100k-\$300k, but a long tail extends to \$755k.

This skew is a problem for regression models. A few very expensive houses can dominate the loss function and pull predictions away from the majority of houses. Let's see if a log transformation helps.

In [ ]:
original_prices = train['SalePrice'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(original_prices, kde=True, ax=axes[0])
axes[0].set_title(f'Before log transform (skew = {original_prices.skew():.2f})')
axes[0].set_xlabel('SalePrice ($)')

sns.histplot(np.log1p(original_prices), kde=True, ax=axes[1])
axes[1].set_title(f'After log transform (skew = {np.log1p(original_prices).skew():.2f})')
axes[1].set_xlabel('log(SalePrice)')

plt.tight_layout()
plt.show()

Log transform brings skewness from ~1.88 down to ~0.12 — much closer to a normal distribution.

We'll apply `np.log1p` to SalePrice during training and reverse it with `np.expm1` at prediction time.

In [ ]:
# Apply for rest of EDA
train['SalePrice'] = np.log1p(train['SalePrice'])

## 3. Missing Values

Let's find out what's missing and try to understand *why* it's missing — that determines how we fill it.

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(14, 5))
missing.plot(kind='bar')
plt.title('Missing values per column')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Columns with missing values: {len(missing)}')

PoolQC has 1453 missing out of 1460 — that's 99.5%. Let's check what the non-null values look like.

In [ ]:
print('PoolQC value counts:')
print(train['PoolQC'].value_counts())
print(f'\nHouses with a pool: {train["PoolQC"].notna().sum()}')

print('\nMiscFeature value counts:')
print(train['MiscFeature'].value_counts())

print('\nAlley value counts:')
print(train['Alley'].value_counts())

Only 7 houses have a pool. The 1453 NaNs in PoolQC don't mean unknown pool quality — they mean **no pool**.

Same logic applies to MiscFeature, Alley, Fence, FireplaceQu, and all Garage/Basement categoricals.
NaN = the feature doesn't exist, not that we don't know its value. These should be filled with "None".

In [ ]:
# Separate categorical vs numeric nulls — different strategies needed
print('Categorical columns with nulls:')
cat_nulls = (train.select_dtypes(include='object')
             .isnull().sum()
             .sort_values(ascending=False)
             .loc[lambda x: x > 0])
print(cat_nulls)

print('\nNumeric columns with nulls:')
num_nulls = (train.select_dtypes(exclude='object')
             .isnull().sum()
             .sort_values(ascending=False)
             .loc[lambda x: x > 0])
print(num_nulls)

LotFrontage is different — 259 missing values that are genuinely unknown, not structural. Let's investigate.

In [ ]:
sns.histplot(train['LotFrontage'].dropna(), kde=True)
plt.title('LotFrontage distribution (non-null values)')
plt.show()

# Does LotFrontage vary by neighborhood?
neighborhood_medians = (train.groupby('Neighborhood')['LotFrontage']
                        .median()
                        .sort_values())

plt.figure(figsize=(14, 5))
neighborhood_medians.plot(kind='bar')
plt.title('Median LotFrontage by Neighborhood')
plt.ylabel('LotFrontage (ft)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Min median: {neighborhood_medians.min()} ft ({neighborhood_medians.idxmin()})')
print(f'Max median: {neighborhood_medians.max()} ft ({neighborhood_medians.idxmax()})')

LotFrontage varies significantly by neighborhood — from 21ft in BrDale to 91ft in NoRidge.

Filling with the global median (70ft) would be inaccurate for houses in neighborhoods with very different lot sizes.
Better approach: fill each missing value with **the median LotFrontage of its neighborhood**.

This needs to be done carefully in the pipeline — the medians must be learned from training data only to avoid leakage.

## 4. Outlier Detection

Let's look at the relationship between living area and price to see if anything looks off.

In [ ]:
sns.scatterplot(x=train['GrLivArea'], y=train['SalePrice'], alpha=0.5)
plt.axvline(x=4000, color='red', linestyle='--', label='Threshold (4000 sqft)', alpha=0.7)
plt.title('GrLivArea vs SalePrice')
plt.xlabel('GrLivArea (sqft)')
plt.ylabel('log(SalePrice)')
plt.legend()
plt.show()

There are a few houses with GrLivArea > 4000 sqft but surprisingly low prices. Let's look at them.

In [ ]:
suspects = train[(train['GrLivArea'] > 4000)][['GrLivArea', 'SalePrice', 'Neighborhood', 'OverallQual']]
suspects['SalePrice_raw'] = np.expm1(suspects['SalePrice'])
print(suspects.sort_values('SalePrice'))

Two houses with over 4000 sqft are selling for under \$200k — that makes no sense for houses of that size.
The other large houses sell for appropriately high prices.

These two (and one more at ~5600 sqft for ~$160k) are likely partial sales, foreclosures, or data errors.
If we keep them, the model will learn a false relationship: very large house = possibly cheap.

Decision: remove houses where GrLivArea > 4000 AND SalePrice < \$200k.

In [ ]:
print('Before:', train.shape)
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 12.5))]
print('After: ', train.shape)

# Confirm they're gone
sns.scatterplot(x=train['GrLivArea'], y=train['SalePrice'], alpha=0.5)
plt.title('GrLivArea vs SalePrice — after outlier removal')
plt.xlabel('GrLivArea (sqft)')
plt.ylabel('log(SalePrice)')
plt.show()

## 5. Correlation Analysis

Which features correlate most with SalePrice? This will guide feature selection and engineering.

In [ ]:
corr = train.corr(numeric_only=True)
print(corr['SalePrice'].sort_values(ascending=False).to_string())

OverallQual leads at 0.82, followed by GrLivArea (0.70) and GarageCars (0.68). Let's visualise the top 15.

In [ ]:
top_features = corr['SalePrice'].abs().sort_values(ascending=False)[1:16].index

plt.figure(figsize=(12, 8))
sns.heatmap(
    train[top_features].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5
)
plt.title('Correlation matrix — top 15 features')
plt.tight_layout()
plt.show()

A few things stand out:
- GarageCars and GarageArea are highly correlated with each other (r = 0.88) — keeping both would be redundant
- YearBuilt and GarageYrBlt are correlated — garages tend to be built with the house
- TotRmsAbvGrd and GrLivArea are correlated — more rooms = more space

We don't need to drop these manually — XGBoost handles multicollinearity well. But it informs our feature engineering decisions.

## 6. Key Feature Relationships

Let's plot the top correlators against SalePrice individually.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pairs = [
    ('OverallQual', 'Overall quality (1-10) vs SalePrice'),
    ('GrLivArea',   'Above ground living area vs SalePrice'),
    ('YearBuilt',   'Year built vs SalePrice'),
    ('GarageCars',  'Garage capacity vs SalePrice'),
]

for ax, (col, title) in zip(axes.flat, pairs):
    ax.scatter(train[col], train['SalePrice'], alpha=0.4)
    ax.set_xlabel(col)
    ax.set_ylabel('log(SalePrice)')
    ax.set_title(title)

plt.tight_layout()
plt.show()

- **OverallQual:** Clear step-wise increase — each quality grade adds roughly equal price increments. Variance widens at higher quality (luxury homes have more price spread)
- **GrLivArea:** Strong linear trend after log transform. Confirms the outlier removal was right — those bottom-right points would have distorted this relationship
- **YearBuilt:** Newer houses command higher prices, but there's high variance in older stock — some old houses are well-maintained and expensive
- **GarageCars:** Step-wise — 0 to 1 car space is a big jump, diminishing returns after 2

## 7. Numeric Feature Distributions

In [ ]:
numeric_cols = ['GrLivArea', 'LotArea', 'TotalBsmtSF',
                'GarageArea', 'YearBuilt', 'LotFrontage']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(train[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)

plt.suptitle('Key numeric feature distributions', y=1.02)
plt.tight_layout()
plt.show()

- GrLivArea and LotArea are right-skewed — a few very large properties
- TotalBsmtSF has a spike at 0 — houses with no basement. This needs to be handled separately from houses with small basements
- YearBuilt peaks post-1950s — most of the dataset is relatively modern housing stock

## 8. Categorical Feature Analysis

In [ ]:
# How many unique values does each categorical column have?
cat_cols = train.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'{col}: {train[col].nunique()} unique values')

This tells us which columns need ordinal encoding (ordered categories like quality ratings) vs one-hot encoding (unordered categories like neighborhood names).

Columns like ExterQual, KitchenQual, BsmtQual have a natural order (Po < Fa < TA < Gd < Ex) — these should be label encoded with a manual mapping, not one-hot encoded.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, ['MSZoning', 'HouseStyle', 'SaleCondition', 'BldgType']):
    train[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

MSZoning is heavily dominated by RL (residential low density). SaleCondition is mostly Normal sales.
These imbalanced distributions mean rare categories will have very few training examples — something to watch for.

## 9. Neighborhood — A Closer Look

Neighborhood has 25 unique values. Before deciding how to encode it, let's see how much price actually varies.

In [ ]:
plt.figure(figsize=(16, 6))
neighborhood_order = (train.groupby('Neighborhood')['SalePrice']
                      .median()
                      .sort_values()
                      .index)

sns.boxplot(x='Neighborhood', y='SalePrice',
            data=train, order=neighborhood_order)
plt.xticks(rotation=45, ha='right')
plt.title('SalePrice distribution by Neighborhood')
plt.ylabel('log(SalePrice)')
plt.tight_layout()
plt.show()

# Raw price range
nbhd_stats = train.groupby('Neighborhood')['SalePrice'].median().sort_values()
print(f'Cheapest neighborhood: {nbhd_stats.index[0]} — median ${np.expm1(nbhd_stats.iloc[0]):,.0f}')
print(f'Most expensive: {nbhd_stats.index[-1]} — median ${np.expm1(nbhd_stats.iloc[-1]):,.0f}')

The price range across neighborhoods is enormous. One-hot encoding Neighborhood would create 25 sparse binary columns — most would be near-zero for any given house.

A better approach: **target encode** — replace each neighborhood with its mean SalePrice. This compresses the signal into a single dense numeric feature.

This needs careful implementation to avoid data leakage — see the pipeline notebook for the out-of-fold encoding approach.

## 10. Summary of Findings

**Target variable**
- SalePrice is right-skewed (skew = 1.88) — log transform applied, reduces to ~0.12
- Predictions must be reverse-transformed with np.expm1 before submission

**Outliers**
- 3 houses removed: GrLivArea > 4000 sqft but SalePrice < \$200k
- Likely data anomalies — keeping them would teach the model a false relationship

**Missing values**
- Most nulls are structural (no pool, no garage, no basement) → fill with "None" or 0
- LotFrontage (259 nulls) genuinely unknown → impute using neighborhood median
- Neighborhood median must be computed from training data only to avoid leakage

**Feature relationships**
- Strongest predictors: OverallQual (0.82), GrLivArea (0.70), GarageCars (0.68)
- GarageCars and GarageArea are redundant (r = 0.88 with each other)
- YearBuilt and YearRemodAdd both positively correlated — age and recency of renovation matter

**Encoding decisions**
- Quality/condition columns (ExterQual, KitchenQual, etc.) → ordinal encoding with manual ordering
- Nominal columns (MSZoning, HouseStyle, etc.) → one-hot encoding
- Neighborhood → target encoding (too much price signal to waste on 25 sparse dummies)

These decisions are all implemented in `02_pipeline.ipynb`.